# Validation Test Evaluation - Capillary Wave

This notebook and the corresponding simulation setup notebook (CapillaryWave_Run.ipynb) are part of published results (cf. 7.3) found in *On a marching level-set method for extended discontinuous Galerkin methods for incompressible two-phase flows: Application to two-dimensional settings* (https://doi.org/10.1002%2Fnme.6853).

In [ ]:
#r "BoSSSpad.dll"
//#r "..\..\..\src\L4-application\BoSSSpad\bin\Release\net8.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
wmg.Init("XNSEpaper_CapillaryWave");
//OpenOrCreateDatabase(@"\\fdygitrunner\ValidationTests\databases\XNSEpaper_CapillaryWave");

Opening existing database '\\fdygitrunner\ValidationTests\databases\XNSEpaper_CapillaryWave'.


In [3]:
using System.IO;
using BoSSS.Application.XNSE_Solver.Logging;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;

## Sessions

In [ ]:
var sessions = wmg.Sessions;
//var sessions = databases.Pick(0).Sessions;
sessions

#0: XNSEpaper_CapillaryWave	CapillaryWave_La3e5_dtfix_convStudy_k3_mesh16	10/24/2025 10:35:38 AM	3513395e...
#1: XNSEpaper_CapillaryWave	CapillaryWave_La3e5_dtfix_convStudy_k2_mesh32	10/24/2025 10:35:07 AM	8986c165...
#2: XNSEpaper_CapillaryWave	CapillaryWave_La3e5_convStudy_k2_mesh32	10/24/2025 10:34:31 AM	b2e9ca1f...
#3: XNSEpaper_CapillaryWave	CapillaryWave_La120_convStudy_k2_mesh32	10/24/2025 10:33:15 AM	3421e749...
#4: XNSEpaper_CapillaryWave	CapillaryWave_La3000_convStudy_k2_mesh32	10/24/2025 10:33:54 AM	b8653a03...
#5: XNSEpaper_CapillaryWave	CapillaryWave_La3e5_dtfix_convStudy_k3_mesh8	10/24/2025 10:35:19 AM	6c210933...
#6: XNSEpaper_CapillaryWave	CapillaryWave_La3e5_dtfix_convStudy_k2_mesh16	10/24/2025 10:34:58 AM	45a474b4...
#7: XNSEpaper_CapillaryWave	CapillaryWave_La3_convStudy_k2_mesh32	10/24/2025 10:32:39 AM	07fa482a...
#8: XNSEpaper_CapillaryWave	CapillaryWave_La3e5_dtfix_convStudy_k2_mesh8	10/24/2025 10:34:44 AM	6775b7c6...
#9: XNSEpaper_CapillaryWave	CapillaryWave_La3e

In [7]:
// foreach (var s in delSessions) {
//     if(s.CreationTime.Date == new DateTime(2025, 10, 20)) {
//         s.Delete(true);
//         //Console.WriteLine($"Deleted session from {s.CreationTime.Date}");
//     }
// }

In [ ]:
// var sessions = wmg.Sessions;
// sessions

In [9]:
string[] LaStudy = { "3", "120", "3000", "3e5", "3e5_dtfix" };
int[] pDegS = { 2, 3 };
int[] Resolutions = { 8, 16, 32 };

In [ ]:
// origin database including all sessions, in case of missing sessions in wmg, due to rerunning only a subset of all needed runs (reduced computational cost)
var originDB = OpenOrCreateDatabase(@"\\fdygitrunner\ValidationTests\databases\bkup-2025Oct14_000000.XNSEpaper_CapillaryWave");

In [11]:
List<ISessionInfo> evalSess = new List<ISessionInfo>();

In [ ]:
foreach (int pDeg in pDegS) {
foreach (string La in LaStudy) {
    if (La != "3e5_dtfix" && pDeg == 3)
        continue;
foreach (int Res in Resolutions) {
    string studyName = $"CapillaryWave_La{La}_convStudy_k{pDeg}_mesh{Res}";
    Console.WriteLine("Searching for: " + studyName);
    var studySess = sessions.Where(sess => sess.Name.Contains(studyName));
    int studyCount = studySess.Count();
    if (studyCount == 0) {
        //Console.WriteLine("Not found in wmg!");
        
        // try to get from originDB
        var originSess = originDB.Sessions.Where(sess => sess.Name.Contains(studyName));
        int originCount = originSess.Count();
        if (originCount == 0) {
            Console.WriteLine("Not found!");
        } else if (originCount > 1) {
            Console.WriteLine("Restart session found in originDB! Take last run");       
            evalSess.Add(originSess.Where(sess => sess.Name.EndsWith($"_restart{originCount-1}")).Single());
        } else {
            evalSess.Add(originSess.Single());
            Console.WriteLine("found in originDB");
        }

    } else if (studyCount > 1) {
        Console.WriteLine("Restart session found! Take last run");       
        evalSess.Add(studySess.Where(sess => sess.Name.EndsWith($"_restart{studyCount-1}")).Single());
    } else {
        evalSess.Add(studySess.Single());
        Console.WriteLine("found");
    }
}
}
}
evalSess

#0: XNSEpaper_CapillaryWave	CapillaryWave_La3_convStudy_k2_mesh8	3/23/2020 9:35:37 PM	bdd58f86...
#1: XNSEpaper_CapillaryWave	CapillaryWave_La3_convStudy_k2_mesh16	3/24/2020 2:29:40 PM	eed03bcf...
#2: XNSEpaper_CapillaryWave	CapillaryWave_La3_convStudy_k2_mesh32	3/26/2020 11:08:20 AM	836becc2...
#3: XNSEpaper_CapillaryWave	CapillaryWave_La120_convStudy_k2_mesh8	3/23/2020 9:34:58 PM	fcfecf85...
#4: XNSEpaper_CapillaryWave	CapillaryWave_La120_convStudy_k2_mesh16	3/24/2020 2:28:51 PM	eecc50f2...
#5: XNSEpaper_CapillaryWave	CapillaryWave_La120_convStudy_k2_mesh32	3/26/2020 11:09:22 AM	031ca6f7...
#6: XNSEpaper_CapillaryWave	CapillaryWave_La3000_convStudy_k2_mesh8	3/18/2020 9:27:41 PM	7126f6b4...
#7: XNSEpaper_CapillaryWave	CapillaryWave_La3000_convStudy_k2_mesh16	3/24/2020 2:28:10 PM	28a39008...
#8: XNSEpaper_CapillaryWave	CapillaryWave_La3000_convStudy_k2_mesh32	3/26/2020 11:09:54 AM	49c7f8d4...
#9: XNSEpaper_CapillaryWave	CapillaryWave_La3e5_convStudy_k2_mesh8	3/23/2020 9:34:01 PM	0af8a8

In [14]:
NUnit.Framework.Assert.AreEqual(18, evalSess.Count, $"Not enough sessions for evaluation.");

### compute times

In [15]:
bool calcComputeTimes = false;

In [16]:
if (calcComputeTimes) {
    
foreach (var sess in evalSess) {
    var nameData = sess.Name.Split(new string[] { "_" }, StringSplitOptions.RemoveEmptyEntries);
    string sessName = "";
    int nameLength = nameData.Length;
    for (int i = 0; i < nameLength; i++) {
        if (nameData[i].StartsWith("restart"))
            continue;
        if (i == 0)
            sessName += nameData[i];
        else    
            sessName += "_" + nameData[i];
    }

    // determine compute time
    Stack<ISessionInfo>  procSIs = new Stack<ISessionInfo>();
    procSIs.Push(sess);
    var currSI = sess;
    var currTimesteps = currSI.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber);
    int timesteps = currTimesteps.Last().TimeStepNumber.MajorNumber;
    double physicalTime = currTimesteps.Last().PhysicalTime;
    TimeSpan computeTime = currTimesteps.Last().CreationTime - currSI.Timesteps.First().CreationTime;
    var rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
    while(!rSIs.IsNullOrEmpty()) {
        var rSI = rSIs.Single();
        procSIs.Push(rSI);
        currSI = rSI;
        currTimesteps = currSI.Timesteps.OrderBy(ts => ts.TimeStepNumber.MajorNumber);
        computeTime = computeTime + (currTimesteps.Last().CreationTime - currTimesteps.First().CreationTime);
        rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
    }

    Console.WriteLine($"Session - {sessName}: timesteps = {timesteps} (t_end = {physicalTime}); compute time = {computeTime.Days}:{computeTime.Hours}");
}
}

### read log data

In [17]:
var evalData = evalSess.ReadLogDataForXNSE(WaveLikeLogging.LogfileName);

In [18]:
public static Plot2Ddata GetAmplitudeOverTime_Plot2Ddata(List<Plot2Ddata> data, string La, string mesh = null) {
    Plot2Ddata plt =  new Plot2Ddata();
    int index = 0;
    plt.Xlabel = "Time";
    plt.Ylabel = "Amplitude height";
    
    var datGroups = data.ElementAt(index).dataGroups;
    int lcIdx = 1;
    for (int i = 0; i < datGroups.Count(); i++) {

        var fmt = new PlotFormat();
        fmt.Style = Styles.Lines; 
        fmt.DashType = DashTypes.Solid;
        fmt.LineWidth = 2;
        fmt.LineColor = (LineColors)(lcIdx); //LineColors.Blue;

        string[] nameData = datGroups[i].Name.Split(new string[] { "-" }, StringSplitOptions.RemoveEmptyEntries);
        if (nameData.Length == 5 && nameData[1] == $"La{La}"
            || nameData.Length == 6 && $"nameData[1]_dtfix" == $"La{La}") {
            if (nameData.Last() == $"mesh{mesh}" || mesh == null) {
                plt.AddDataGroup(nameData.Last(), datGroups[i].Abscissas, datGroups[i].Values, fmt);
                lcIdx++;
            }
        }
    }

    plt.Title = $"La={La}";
    plt.ShowLegend = true;

    return plt;
}

In [19]:
GetAmplitudeOverTime_Plot2Ddata(evalData, "3").ToGnuplot().PlotSVG()

<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 0.001 
 
 
 
 
 0.002 
 
 
 
 
 0.003 
 
 
 
 
 0.004 
 
 
 
 
 0.005 
 
 
 
 
 0.006 
 
 
 
 
 0.007 
 
 
 
 
 0.008 
 
 
 
 
 0.009 
 
 
 
 
 0.01 
 
 
 
 
 0 
 
 
 
 
 0.2 
 
 
 
 
 0.4 
 
 
 
 
 0.6 
 
 
 
 
 0.8 
 
 
 
 
 1 
 
 
 
 
 1.2 
 
 
 
 
 1.4 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Amplitude height 
 
 
 
 
 Time 
 
 
 
 
 La=3 
 
 
 mesh8 
 
 
 
 
 mesh8 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M610.7,93.1 L664.1,93.1 M88.5,73.5 L89.4,73.5 L90.4,74.0 L91.3,75.0 L92.3,76.3 L93.2,77.9
 L94.2,79.8 L95.1,81.9 L96.1,84.2 L97.0,86.8 L98.0,89.4 L98.9,92.2 L99.8,95.2 L100.8,98.2
 L101.7,101.3 L102.7,104.5 L103.6,107.8 L104.6,111.2 L105.5,114.6 L106.5,118.0 L107.4,121.5 L108.3,125.0
 L109.3,128.5 L110.2,132.0 L111.2,135.6 L112.1,139.2 L113.1,142.8 L114.0,146.3 L115.0,149.9 L115.9,153.5
 L116.9,157.1 L117.8,160.6 L118.7,164.2 L119.7,167.7 L120.6,171.2 L121.6,174.7 L122.5,178.2 L123.5,181.7
 L124.4,185.1 L125.4,188.6 L126.3,192.0 L127.3,195.4 L128.2,198.7 L129.1,202.1 L130.1,205.4 L131.0,208.7
 L132.0,211.9 L132.9,215.2 L133.9,218.4 L134.8,221.6 L135.8,224.7 L136.7,227.9 L137.6,231.0 L138.6,234.1
 L139.5,237.1 L140.5,240.2 L141.4,243.2 L142.4,246.1 L143.3,249.1 L144.3,252.0 L145.2,254.9 L146.2,257.7
 L147.1,260.6 L148.0,263.4 L149.0,266.2 L149.9,268.9 L150.9,271.7 L151.8,274.4 L152.8,277.1 L153.7,279.7
 L154.7,282.3 L155.6,284.9 L156.6,287.5 L157.5,290.1 L158.4,292.6 L159.4,295.1 L160.3,297.6 L161.3,300.0
 L162.2,302.4 L163.2,304.8 L164.1,307.2 L165.1,309.6 L166.0,311.9 L166.9,314.2 L167.9,316.5 L168.8,318.8
 L169.8,321.0 L170.7,323.2 L171.7,325.4 L172.6,327.6 L173.6,329.8 L174.5,331.9 L175.5,334.0 L176.4,336.1
 L177.3,338.2 L178.3,340.2 L179.2,342.2 L180.2,344.2 L181.1,346.2 L182.1,348.2 L183.0,350.1 L184.0,352.1
 L184.9,354.0 L185.8,355.9 L186.8,357.7 L187.7,359.6 L188.7,361.4 L189.6,363.2 L190.6,365.0 L191.5,366.8
 L192.5,368.5 L193.4,370.3 L194.4,372.0 L195.3,373.7 L196.2,375.4 L197.2,377.1 L198.1,378.7 L199.1,380.4
 L200.0,382.0 L201.0,383.6 L201.9,385.2 L202.9,386.7 L203.8,388.3 L204.8,389.8 L205.7,391.4 L206.6,392.9
 L207.6,394.4 L208.5,395.9 L209.5,397.3 L210.4,398.8 L211.4,400.2 L212.3,401.6 L213.3,403.0 L214.2,404.4
 L215.1,405.8 L216.1,407.2 L217.0,408.5 L218.0,409.9 L218.9,411.2 L219.9,412.5 L220.8,413.8 L221.8,415.1
 L222.7,416.3 L223.7,417.6 L224.6,418.8 L225.5,420.1 L226.5,421.3 L227.4,422.5 L228.4,423.7 L229.3,424.9
 L230.3,426.1 L231.2,427.2 L232.2,428.4 L233.1,429.5 L234.1,430.6 L235.0,431.7 L235.9,432.8 L236.9,433.9
 L237.8,435.0 L238.8,436.1 L239.7,437.1 L240.7,438.2 L241.6,439.2 L242.6,440.3 L243.5,441.3 L244.4,442.3
 L245.4,443.3 L246.3,444.3 L247.3,445.2 L248.2,446.2 L249.2,447.2 L250.1,448.1 L251.1,449.1 L252.0,450.0
 L253.0,450.9 L253.9,451.8 L254.8,452.7 L255.8,453.6 L256.7,454.5 L257.7,455.4 L258.6,456.2 L259.6,457.1
 L260.5,457.9 L261.5,458.8 L262.4,459.6 L263.4,460.4 L264.3,461.2 L265.2,462.0 L266.2,462.8 L267.1,463.6
 L268.1,464.4 L269.0,465.2 L270.0,466.0 L270.9,466.7 L271.9,467.5 L272.8,468.2 L273.7,469.0 L274.7,469.7
 L275.6,470.4 L276.6,471.1 L277.5,471.8 L278.5,472.5 L279.4,473.2 L280.4,473.9 L281.3,474.6 L282.3,475.3
 L283.2,475.9 L284.1,476.6 L285.1,477.2 L286.0,477.9 L287.0,478.5 L287.9,479.2 L288.9,479.8 L289.8,480.4
 L290.8,481.0 L291.7,481.6 L292.7,482.2 L293.6,482.8 L294.5,483.4 L295.5,484.0 L296.4,484.6 L297.4,485.1
 L298.3,485.7 L299.3,486.3 L300.2,486.8 L301.2,487.4 L302.1,487.9 L303.0,488.5 L304.0,489.0 L304.9,489.5
 L305.9,490.0 L306.8,490.6 L307.8,491.1 L308.7,491.6 L309.7,492.1 L310.6,492.6 L311.6,493.1 L312.5,493.6
 L313.4,494.

In [20]:
GetAmplitudeOverTime_Plot2Ddata(evalData, "120").ToGnuplot().PlotSVG()

<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 0.001 
 
 
 
 
 0.002 
 
 
 
 
 0.003 
 
 
 
 
 0.004 
 
 
 
 
 0.005 
 
 
 
 
 0.006 
 
 
 
 
 0.007 
 
 
 
 
 0.008 
 
 
 
 
 0.009 
 
 
 
 
 0.01 
 
 
 
 
 0 
 
 
 
 
 0.05 
 
 
 
 
 0.1 
 
 
 
 
 0.15 
 
 
 
 
 0.2 
 
 
 
 
 0.25 
 
 
 
 
 0.3 
 
 
 
 
 0.35 
 
 
 
 
 0.4 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Amplitude height 
 
 
 
 
 Time 
 
 
 
 
 La=120 
 
 
 mesh8 
 
 
 
 
 mesh8 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M610.7,93.1 L664.1,93.1 M88.5,73.5 L101.7,73.5 L115.0,142.4 L128.2,259.3 L141.4,399.4 L154.7,538.0
 L167.9,430.5 L181.1,349.8 L194.4,310.8 L207.6,312.2 L220.8,347.0 L234.1,404.2 L247.3,471.6 L260.5,537.6
 L273.7,492.1 L287.0,454.0 L300.2,435.5 L313.4,436.1 L326.7,452.2 L339.9,478.8 L353.1,510.1 L366.4,540.8
 L379.6,518.5 L392.8,500.8 L406.1,492.4 L419.3,492.7 L432.5,500.3 L445.8,512.8 L459.0,527.4 L472.2,541.7
 L485.5,531.2 L498.7,523.0 L511.9,519.1 L525.2,519.3 L538.4,522.8 L551.6,528.7 L564.9,535.5 L578.1,542.2
 L591.3,537.1 L604.5,533.3 L617.8,531.5 L631.0,531.6 L644.2,533.3 L657.5,536.0 L670.7,539.2 L683.9,542.3
 L697.2,539.9 L710.4,538.1 L723.6,537.3 L736.9,537.4 L750.1,538.1 '/> 
 
 mesh16 
 
 
 mesh16 
 
 
 
	<path stroke='rgb( 0, 255, 0)' d='M610.7,117.1 L664.1,117.1 M88.5,72.5 L89.0,72.5 L89.5,72.6 L90.0,72.9 L90.5,73.3 L91.0,73.9
 L91.5,74.5 L92.0,75.3 L92.5,76.3 L93.0,77.3 L93.5,78.5 L94.0,79.8 L94.5,81.1 L95.0,82.6
 L95.4,84.2 L95.9,85.9 L96.4,87.7 L96.9,89.7 L97.4,91.7 L97.9,93.8 L98.4,96.0 L98.9,98.3
 L99.4,100.6 L99.9,103.1 L100.4,105.7 L100.9,108.3 L101.4,111.0 L101.9,113.8 L102.4,116.7 L102.9,119.7
 L103.4,122.7 L103.9,125.9 L104.4,129.0 L104.9,132.3 L105.4,135.6 L105.9,139.0 L106.4,142.5 L106.9,146.0
 L107.4,149.6 L107.9,153.3 L108.3,157.0 L108.8,160.7 L109.3,164.6 L109.8,168.5 L110.3,172.4 L110.8,176.4
 L111.3,180.4 L111.8,184.5 L112.3,188.6 L112.8,192.8 L113.3,197.0 L113.8,201.3 L114.3,205.6 L114.8,209.9
 L115.3,214.3 L115.8,218.7 L116.3,223.1 L116.8,227.6 L117.3,232.1 L117.8,236.7 L118.3,241.2 L118.8,245.8
 L119.3,250.5 L119.8,255.1 L120.3,259.8 L120.8,264.5 L121.2,269.2 L121.7,273.9 L122.2,278.6 L122.7,283.4
 L123.2,288.2 L123.7,293.0 L124.2,297.8 L124.7,302.6 L125.2,307.4 L125.7,312.3 L126.2,317.1 L126.7,321.9
 L127.2,326.8 L127.7,331.6 L128.2,336.5 L128.7,341.4 L129.2,346.2 L129.7,351.1 L130.2,355.9 L130.7,360.8
 L131.2,365.6 L131.7,370.5 L132.2,375.3 L132.7,380.1 L133.2,384.9 L133.7,389.7 L134.2,394.5 L134.6,399.3
 L135.1,404.1 L135.6,408.8 L136.1,413.6 L136.6,418.3 L137.1,423.0 L137.6,427.7 L138.1,432.4 L138.6,437.0
 L139.1,441.6 L139.6,446.2 L140.1,450.8 L140.6,455.4 L141.1,459.9 L141.6,464.4 L142.1,468.9 L142.6,473.4
 L143.1,477.8 L143.6,482.2 L144.1,486.6 L144.6,490.9 L145.1,495.2 L145.6,499.5 L146.1,503.7 L146.6,508.0
 L147.1,512.1 L147.5,516.3 L148.0,520.4 L148.5,524.5 L149.0,528.5 L149.5,532.5 L150.0,536.5 L150.5,540.4
 L151.0,540.5 L151.5,536.6 L152.0,532.8 L152.5,529.0 L153.0,525.3 L153.5,521.6 L154.0,517.9 L154.5,514.3
 L155.0,510.8 L155.5,507.2 L156.0,503.7 L156.5,500.3 L157.0,496.9 L157.5,493.5 L158.0,490.2 L158.5,487.0
 L159.0,483.7 L159.5,480.6 L160.0,477.4 L160.4,474.3 L160.9,471.3 L161.4,468.3 L161.9,465.4 L162.4,462.5
 L162.9,459.6 L163.4,456.8 L163.9,454.0 L164.4,451.3 L164.9,448.7 L165.4,446.0 L165.9,443.5 L166.4,441.0
 L166.9,438.5 L167.4,436.1 L167.9,433.7 L168.4,431.3 L168.9,429.1 L169.4,426.8 L169.9,424.6 L170.4,422.5
 L170.9,420.4 L171.4,418.4 L171.9,416.4 L172.4,414.5 L172.9,412.6 L173.4,410.7 L173.8,408.9 L174.3,407.2
 L174.8,405.5 L175.3,403.8 L175.8,402.2 L176.3,400.7 L176.8,399.1 L177.3,397.

In [21]:
GetAmplitudeOverTime_Plot2Ddata(evalData, "3000").ToGnuplot().PlotSVG()

<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 0.001 
 
 
 
 
 0.002 
 
 
 
 
 0.003 
 
 
 
 
 0.004 
 
 
 
 
 0.005 
 
 
 
 
 0.006 
 
 
 
 
 0.007 
 
 
 
 
 0.008 
 
 
 
 
 0.009 
 
 
 
 
 0.01 
 
 
 
 
 0 
 
 
 
 
 0.05 
 
 
 
 
 0.1 
 
 
 
 
 0.15 
 
 
 
 
 0.2 
 
 
 
 
 0.25 
 
 
 
 
 0.3 
 
 
 
 
 0.35 
 
 
 
 
 0.4 
 
 
 
 
 0.45 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Amplitude height 
 
 
 
 
 Time 
 
 
 
 
 La=3000 
 
 
 mesh8 
 
 
 
 
 mesh8 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M610.7,93.1 L664.1,93.1 M88.5,73.5 L89.7,73.5 L90.9,74.6 L92.0,76.6 L93.2,79.7 L94.4,83.6
 L95.6,88.5 L96.7,94.3 L97.9,101.0 L99.1,108.5 L100.3,116.8 L101.4,126.0 L102.6,135.9 L103.8,146.6
 L105.0,157.9 L106.1,170.0 L107.3,182.7 L108.5,196.0 L109.7,209.9 L110.8,224.3 L112.0,239.3 L113.2,254.8
 L114.4,270.7 L115.6,287.0 L116.7,303.7 L117.9,320.7 L119.1,338.0 L120.3,355.6 L121.4,373.4 L122.6,391.4
 L123.8,409.5 L125.0,427.8 L126.1,446.1 L127.3,464.5 L128.5,482.8 L129.7,501.2 L130.8,519.4 L132.0,537.5
 L133.2,529.3 L134.4,511.4 L135.5,493.8 L136.7,476.5 L137.9,459.4 L139.1,442.6 L140.3,426.1 L141.4,410.1
 L142.6,394.4 L143.8,379.1 L145.0,364.3 L146.1,350.0 L147.3,336.1 L148.5,322.8 L149.7,310.0 L150.8,297.8
 L152.0,286.2 L153.2,275.2 L154.4,264.7 L155.5,255.0 L156.7,245.8 L157.9,237.3 L159.1,229.5 L160.2,222.4
 L161.4,215.9 L162.6,210.1 L163.8,205.1 L165.0,200.7 L166.1,197.0 L167.3,194.0 L168.5,191.8 L169.7,190.2
 L170.8,189.3 L172.0,189.1 L173.2,189.6 L174.4,190.8 L175.5,192.7 L176.7,195.2 L177.9,198.4 L179.1,202.1
 L180.2,206.6 L181.4,211.6 L182.6,217.2 L183.8,223.4 L184.9,230.2 L186.1,237.5 L187.3,245.3 L188.5,253.7
 L189.7,262.5 L190.8,271.8 L192.0,281.5 L193.2,291.6 L194.4,302.1 L195.5,313.0 L196.7,324.2 L197.9,335.7
 L199.1,347.6 L200.2,359.7 L201.4,372.0 L202.6,384.6 L203.8,397.3 L204.9,410.2 L206.1,423.2 L207.3,436.4
 L208.5,449.6 L209.6,462.9 L210.8,476.2 L212.0,489.5 L213.2,502.8 L214.4,516.0 L215.5,529.2 L216.7,542.2
 L217.9,529.7 L219.1,516.9 L220.2,504.3 L221.4,491.9 L222.6,479.7 L223.8,467.7 L224.9,456.0 L226.1,444.6
 L227.3,433.5 L228.5,422.7 L229.6,412.2 L230.8,402.1 L232.0,392.4 L233.2,383.1 L234.3,374.1 L235.5,365.6
 L236.7,357.5 L237.9,349.9 L239.1,342.7 L240.2,335.9 L241.4,329.7 L242.6,323.9 L243.8,318.6 L244.9,313.8
 L246.1,309.4 L247.3,305.6 L248.5,302.3 L249.6,299.5 L250.8,297.2 L252.0,295.4 L253.2,294.1 L254.3,293.2
 L255.5,292.9 L256.7,293.1 L257.9,293.8 L259.0,295.0 L260.2,296.6 L261.4,298.7 L262.6,301.2 L263.8,304.3
 L264.9,307.7 L266.1,311.6 L267.3,315.9 L268.5,320.6 L269.6,325.7 L270.8,331.1 L272.0,337.0 L273.2,343.2
 L274.3,349.7 L275.5,356.5 L276.7,363.7 L277.9,371.1 L279.0,378.8 L280.2,386.7 L281.4,394.9 L282.6,403.3
 L283.7,411.9 L284.9,420.6 L286.1,429.5 L287.3,438.6 L288.5,447.8 L289.6,457.1 L290.8,466.4 L292.0,475.9
 L293.2,485.3 L294.3,494.8 L295.5,504.3 L296.7,513.8 L297.9,523.3 L299.0,532.7 L300.2,542.0 L301.4,533.6
 L302.6,524.4 L303.7,515.4 L304.9,506.5 L306.1,497.7 L307.3,489.2 L308.4,480.8 L309.6,472.6 L310.8,464.6
 L312.0,456.8 L313.1,449.3 L314.3,442.1 L315.5,435.1 L316.7,428.3 L317.9,421.9 L319.0,415.7 L320.2,409.9
 L321.4,404.4 L322.6,399.2 L323.7,394.3 L324.9,389.7 L326.1,385.5 L327.3,381.7 L328.4,378.1 L329.6,375.0
 L330.8,372.2 L332.0,369.7 L333.1,367.7 L334.3,365.9 L335.5,364.6 L336.7,363.6 L337.8,362.9 L339.0,362.7
 L340.2,362.7 L341.4,363.1 L342.6,363.9 L343.7,365.0 L344.9,366.4 L346.1,368.2 L347.3,370.3 L348.4,372.7
 L349.6,375.4 L350.8,378.5 L352.0,381.8 L353.1,385.4 L354.3,389.2 L355.5,393.4 L356.7,397.8 L357.8,402.4
 L359.0,407.2 L360.2,412.3 L361.4,417.6 L362.5,423

In [22]:
GetAmplitudeOverTime_Plot2Ddata(evalData, "3e5").ToGnuplot().PlotSVG()

<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 0.001 
 
 
 
 
 0.002 
 
 
 
 
 0.003 
 
 
 
 
 0.004 
 
 
 
 
 0.005 
 
 
 
 
 0.006 
 
 
 
 
 0.007 
 
 
 
 
 0.008 
 
 
 
 
 0.009 
 
 
 
 
 0.01 
 
 
 
 
 0 
 
 
 
 
 0.05 
 
 
 
 
 0.1 
 
 
 
 
 0.15 
 
 
 
 
 0.2 
 
 
 
 
 0.25 
 
 
 
 
 0.3 
 
 
 
 
 0.35 
 
 
 
 
 0.4 
 
 
 
 
 0.45 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Amplitude height 
 
 
 
 
 Time 
 
 
 
 
 La=3e5 
 
 
 mesh8 
 
 
 
 
 mesh8 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M610.7,93.1 L664.1,93.1 M88.5,73.5 L89.7,73.5 L90.9,74.7 L92.0,76.9 L93.2,80.1 L94.4,84.5
 L95.6,89.8 L96.7,96.2 L97.9,103.6 L99.1,112.0 L100.3,121.3 L101.4,131.6 L102.6,142.7 L103.8,154.7
 L105.0,167.6 L106.1,181.2 L107.3,195.7 L108.5,210.9 L109.7,226.8 L110.8,243.4 L112.0,260.6 L113.2,278.4
 L114.4,296.7 L115.6,315.5 L116.7,334.8 L117.9,354.6 L119.1,374.7 L120.3,395.1 L121.4,415.8 L122.6,436.7
 L123.8,457.9 L125.0,479.2 L126.1,500.5 L127.3,522.0 L128.5,541.4 L129.7,520.1 L130.8,498.8 L132.0,477.6
 L133.2,456.7 L134.4,435.9 L135.5,415.5 L136.7,395.4 L137.9,375.6 L139.1,356.2 L140.3,337.3 L141.4,318.9
 L142.6,301.0 L143.8,283.7 L145.0,267.0 L146.1,250.9 L147.3,235.5 L148.5,220.8 L149.7,206.8 L150.8,193.6
 L152.0,181.2 L153.2,169.6 L154.4,158.8 L155.5,148.9 L156.7,139.9 L157.9,131.8 L159.1,124.5 L160.2,118.3
 L161.4,112.9 L162.6,108.5 L163.8,105.1 L165.0,102.6 L166.1,101.0 L167.3,100.5 L168.5,100.9 L169.7,102.3
 L170.8,104.6 L172.0,107.8 L173.2,112.0 L174.4,117.2 L175.5,123.2 L176.7,130.1 L177.9,138.0 L179.1,146.6
 L180.2,156.1 L181.4,166.5 L182.6,177.6 L183.8,189.5 L184.9,202.1 L186.1,215.5 L187.3,229.5 L188.5,244.2
 L189.7,259.4 L190.8,275.3 L192.0,291.7 L193.2,308.6 L194.4,325.9 L195.5,343.7 L196.7,361.9 L197.9,380.4
 L199.1,399.3 L200.2,418.4 L201.4,437.7 L202.6,457.2 L203.8,476.8 L204.9,496.6 L206.1,516.3 L207.3,536.1
 L208.5,528.9 L209.6,509.3 L210.8,489.7 L212.0,470.3 L213.2,451.2 L214.4,432.2 L215.5,413.6 L216.7,395.3
 L217.9,377.3 L219.1,359.8 L220.2,342.7 L221.4,326.1 L222.6,310.0 L223.8,294.5 L224.9,279.5 L226.1,265.1
 L227.3,251.4 L228.5,238.4 L229.6,226.0 L230.8,214.4 L232.0,203.5 L233.2,193.4 L234.3,184.1 L235.5,175.6
 L236.7,167.9 L237.9,161.1 L239.1,155.1 L240.2,149.9 L241.4,145.6 L242.6,142.2 L243.8,139.7 L244.9,138.1
 L246.1,137.4 L247.3,137.5 L248.5,138.5 L249.6,140.5 L250.8,143.2 L252.0,146.9 L253.2,151.4 L254.3,156.7
 L255.5,162.9 L256.7,169.9 L257.9,177.7 L259.0,186.2 L260.2,195.5 L261.4,205.6 L262.6,216.3 L263.8,227.8
 L264.9,239.9 L266.1,252.6 L267.3,266.0 L268.5,279.9 L269.6,294.3 L270.8,309.3 L272.0,324.7 L273.2,340.5
 L274.3,356.8 L275.5,373.4 L276.7,390.4 L277.9,407.6 L279.0,425.1 L280.2,442.8 L281.4,460.7 L282.6,478.7
 L283.7,496.8 L284.9,515.0 L286.1,533.2 L287.3,533.5 L288.5,515.4 L289.6,497.4 L290.8,479.6 L292.0,461.9
 L293.2,444.5 L294.3,427.3 L295.5,410.4 L296.7,393.8 L297.9,377.6 L299.0,361.8 L300.2,346.5 L301.4,331.6
 L302.6,317.2 L303.7,303.3 L304.9,290.0 L306.1,277.3 L307.3,265.1 L308.4,253.6 L309.6,242.8 L310.8,232.7
 L312.0,223.2 L313.1,214.5 L314.3,206.5 L315.5,199.3 L316.7,192.8 L317.9,187.1 L319.0,182.2 L320.2,178.1
 L321.4,174.8 L322.6,172.3 L323.7,170.7 L324.9,169.8 L326.1,169.8 L327.3,170.5 L328.4,172.1 L329.6,174.5
 L330.8,177.7 L332.0,181.6 L333.1,186.4 L334.3,191.9 L335.5,198.1 L336.7,205.1 L337.8,212.8 L339.0,221.2
 L340.2,230.3 L341.4,240.1 L342.6,250.5 L343.7,261.5 L344.9,273.0 L346.1,285.2 L347.3,297.8 L348.4,311.0
 L349.6,324.6 L350.8,338.7 L352.0,353.2 L353.1,368.1 L354.3,383.3 L355.5,398.8 L356.7,414.6 L357.8,430.6
 L359.0,446.8 L360.2,463.2 L361.4,479.8 L362.5,496.

## Compare to Reference data (Prosperetti (1981)) - cf. FIGURE 4

In [23]:
string[] refData = new string[]{ "La3", "La120", "La3000", "La3e5" };

In [24]:
Plot2Ddata[] dataRef = new Plot2Ddata[refData.Length];
for (int j = 0; j < refData.Length; j++) {
    // Read all data
    string dat  = "CWrefDatConv_"+refData[j]+".txt";
    string[] lines = File.ReadAllLines(dat);
    double[] time = new double[lines.Length];
    double[] val = new double[lines.Length];

    for (int i = 0; i < lines.Length; i++) {
        time[i] = Convert.ToDouble(lines[i].Split(new string[] { "," }, StringSplitOptions.RemoveEmptyEntries)[0]);            
        val[i] = Math.Abs(Convert.ToDouble(lines[i].Split(new string[] { "," }, StringSplitOptions.RemoveEmptyEntries)[1]));
    }        
    // Build DataSet
    dataRef[j] = new Plot2Ddata(new KeyValuePair<string, double[][]>(refData[j], new double[][] { time, val }));
}

In [25]:
public static Plot2Ddata GetAmplitudeComparisonOverTime_Plot2Ddata(List<Plot2Ddata> data, Plot2Ddata[] refData, string La) {

    Plot2Ddata plt = GetAmplitudeOverTime_Plot2Ddata(data, La, "32");   // comparing to finest mesh

    for (int i = 0; i < refData.Length; i++) {
        var fmt = new PlotFormat();
        fmt.Style = Styles.Lines; 
        fmt.DashType = DashTypes.DotDashed;
        fmt.LineWidth = 2;
        fmt.LineColor = LineColors.Black;
        
        // Add reference data
        if (refData[i].dataGroups[0].Name == $"La{La}")
            plt.AddDataGroup(refData[i].dataGroups[0], fmt);
    }
    
    plt.XrangeMin = 0.0;
    plt.XrangeMax = 0.4;

    return plt;
}

In [26]:
Plot2Ddata[,] Fig4 = new Plot2Ddata[2, 2];

Fig4[0, 0] = GetAmplitudeComparisonOverTime_Plot2Ddata(evalData, dataRef, "3");
Fig4[0, 1] = GetAmplitudeComparisonOverTime_Plot2Ddata(evalData, dataRef, "120");
Fig4[1, 0] = GetAmplitudeComparisonOverTime_Plot2Ddata(evalData, dataRef, "3000");
Fig4[1, 1] = GetAmplitudeComparisonOverTime_Plot2Ddata(evalData, dataRef, "3e5");

Fig4.ToGnuplot().PlotSVG(xRes:1200,yRes:800)

<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.001 
 
 
 
 
 0.002 
 
 
 
 
 0.003 
 
 
 
 
 0.004 
 
 
 
 
 0.005 
 
 
 
 
 0.006 
 
 
 
 
 0.007 
 
 
 
 
 0.008 
 
 
 
 
 0.009 
 
 
 
 
 0.01 
 
 
 
 
 0 
 
 
 
 
 0.05 
 
 
 
 
 0.1 
 
 
 
 
 0.15 
 
 
 
 
 0.2 
 
 
 
 
 0.25 
 
 
 
 
 0.3 
 
 
 
 
 0.35 
 
 
 
 
 0.4 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Amplitude height 
 
 
 
 
 Time 
 
 
 
 
 La=3 
 
 
 mesh32 
 
 
 
 
 mesh32 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M404.7,97.1 L458.1,97.1 M94.4,76.2 L94.7,76.2 L95.1,76.2 L95.4,76.2 L95.7,76.2 L96.1,76.2
 L96.4,76.3 L96.8,76.3 L97.1,76.4 L97.4,76.4 L97.8,76.5 L98.1,76.6 L98.4,76.7 L98.8,76.7
 L99.1,76.8 L99.5,76.9 L99.8,77.0 L100.1,77.1 L100.5,77.2 L100.8,77.4 L101.1,77.5 L101.5,77.6
 L101.8,77.7 L102.2,77.9 L102.5,78.0 L102.8,78.2 L103.2,78.3 L103.5,78.5 L103.8,78.6 L104.2,78.8
 L104.5,78.9 L104.9,79.1 L105.2,79.3 L105.5,79.5 L105.9,79.6 L106.2,79.8 L106.5,80.0 L106.9,80.2
 L107.2,80.4 L107.6,80.6 L107.9,80.8 L108.2,81.0 L108.6,81.2 L108.9,81.4 L109.2,81.6 L109.6,81.8
 L109.9,82.1 L110.3,82.3 L110.6,82.5 L110.9,82.7 L111.3,83.0 L111.6,83.2 L111.9,83.4 L112.3,83.7
 L112.6,83.9 L113.0,84.2 L113.3,84.4 L113.6,84.6 L114.0,84.9 L114.3,85.1 L114.6,85.4 L115.0,85.6
 L115.3,85.9 L115.6,86.2 L116.0,86.4 L116.3,86.7 L116.7,87.0 L117.0,87.2 L117.3,87.5 L117.7,87.8
 L118.0,88.0 L118.3,88.3 L118.7,88.6 L119.0,88.9 L119.4,89.1 L119.7,89.4 L120.0,89.7 L120.4,90.0
 L120.7,90.3 L121.0,90.6 L121.4,90.9 L121.7,91.1 L122.1,91.4 L122.4,91.7 L122.7,92.0 L123.1,92.3
 L123.4,92.6 L123.7,92.9 L124.1,93.2 L124.4,93.5 L124.8,93.8 L125.1,94.1 L125.4,94.4 L125.8,94.7
 L126.1,95.0 L126.4,95.3 L126.8,95.7 L127.1,96.0 L127.5,96.3 L127.8,96.6 L128.1,96.9 L128.5,97.2
 L128.8,97.5 L129.1,97.8 L129.5,98.2 L129.8,98.5 L130.2,98.8 L130.5,99.1 L130.8,99.4 L131.2,99.7
 L131.5,100.1 L131.8,100.4 L132.2,100.7 L132.5,101.0 L132.8,101.4 L133.2,101.7 L133.5,102.0 L133.9,102.3
 L134.2,102.7 L134.5,103.0 L134.9,103.3 L135.2,103.6 L135.5,104.0 L135.9,104.3 L136.2,104.6 L136.6,105.0
 L136.9,105.3 L137.2,105.6 L137.6,106.0 L137.9,106.3 L138.2,106.6 L138.6,107.0 L138.9,107.3 L139.3,107.6
 L139.6,108.0 L139.9,108.3 L140.3,108.6 L140.6,109.0 L140.9,109.3 L141.3,109.6 L141.6,110.0 L142.0,110.3
 L142.3,110.6 L142.6,111.0 L143.0,111.3 L143.3,111.7 L143.6,112.0 L144.0,112.3 L144.3,112.7 L144.7,113.0
 L145.0,113.4 L145.3,113.7 L145.7,114.0 L146.0,114.4 L146.3,114.7 L146.7,115.1 L147.0,115.4 L147.4,115.7
 L147.7,116.1 L148.0,116.4 L148.4,116.8 L148.7,117.1 L149.0,117.4 L149.4,117.8 L149.7,118.1 L150.1,118.5
 L150.4,118.8 L150.7,119.1 L151.1,119.5 L151.4,119.8 L151.7,120.2 L152.1,120.5 L152.4,120.9 L152.7,121.2
 L153.1,121.5 L153.4,121.9 L153.8,122.2 L154.1,122.6 L154.4,122.9 L154.8,123.3 L155.1,123.6 L155.4,123.9
 L155.8,124.3 L156.1,124.6 L156.5,125.0 L156.8,125.3 L157.1,125.6 L157.5,126.0 L157.8,126.3 L158.1,126.7
 L158.5,127.0 L158.8,127.3 L159.2,127.7 L159.5,128.0 L159.8,128.4 L160.2,128.7 L160.5,129.1 L160.8,129.4
 L161.2,129.7 L161.5,130.1 L161.9,130.4 L162.2,130.8 L162.5,131.1 L162.9,131.4 L163.2,131.8 L163.5,132.1
 L163.9,132.5 L164.2,132.8 L164.6,133.1 L164.9,133.5 L165.2,133.8 L165.6,134.1 L165.9,134.5 L166.2,134.8
 L166.6,135.2 L166.9,135.5 L167.3,135.8 L167.6,136.2 L167.9,136.5 L168.3,136.8 L168.6,137.2 L168.9,137.5
 L169.3,137.9 L169.6,138.2 L169.9,138.5 L170.3,138.9 L170.6,139.2 L171.0,139.5 L171.3,139.9 L171.6,140.2
 L172.0,140.5 L172.3,140.9 L172.6,141.2 L173.0,141.5 L173.3,141.9 L173.7,142.2 L174.0,142.5 L174.3,142.9
 L174.7,143.2 L175.0,143.5 L175.3,143.9 L175.7,144.2 L176.0,144.5 L176.4,144.9 L176.7,145.2 L177.0,145

## Convergence evaluation 

In [27]:
public static Plot2Ddata GetErrorNormForConvergence_Plot2Ddata(List<Plot2Ddata> data, List<ISessionInfo> dataSess, Plot2Ddata[] refData, string study, int pDeg = 2) {

    int index = 0;
    //studyName = $"{study}_convStudy";

    List<Plot2Ddata> studyData = new List<Plot2Ddata>();
    Plot2Ddata plt = new Plot2Ddata();
    foreach (var dgrp in data.ElementAt(index).dataGroups) {
        if (dgrp.Name.Contains($"{study.Replace("_", "-")}-convStudy-k{pDeg}")) {
            plt.AddDataGroup(dgrp);
        }
    }
    studyData.Add(plt);
    //Console.WriteLine($"Found {plt.dataGroups.Count()} data sets for convergence study {study} with pDeg {pDeg}.");

    double[] abscissa = dataSess.Where(sess => sess.Name.Contains($"{study}_convStudy_k{pDeg}")).Select(s => Convert.ToDouble(s.KeysAndQueries["Grid:hMax"])).ToArray();
    //Console.WriteLine($"Found {abscissa.Length} hMax values for convergence study {study} with pDeg {pDeg}.");

    string[] nameData = study.Split(new string[] { "_" }, StringSplitOptions.RemoveEmptyEntries);
    double[] refAbs = refData.Where(plt => plt.dataGroups[0].Name == nameData[0]).ElementAt(0).dataGroups[0].Abscissas;
    double[] refVal = refData.Where(plt => plt.dataGroups[0].Name == nameData[0]).ElementAt(0).dataGroups[0].Values;

    List<double> refAbsList = new List<double>();
    List<double> refValList = new List<double>();
    int numRefVal = refAbs.Length;
    for(int p = 0; p < numRefVal; p++) {
        if(refAbs[p] <= 0.4) {   // only consider reference data up to t = 0.4s
            refAbsList.Add(refAbs[p]);
            refValList.Add(refVal[p]);
        }
    }

    var convData = ISessionInfoExtensions.LogDataToConvergenceData(studyData, abscissa, refAbsList.ToArray(), refValList.ToArray());

    return convData.ElementAt(0);
}

In [43]:
public static void Checkl2errorNorms(Plot2Ddata errNorms, double[] refl2Values, double[] refl2Values_err, double refEOC, double refEOC_err) {

    string[] meshSize = new string[] { "8", "16", "32"};
    if (refl2Values.Length != refl2Values_err.Length)
        throw new ArgumentException("Length of reference l2-error values and error values must be the same.");

    for (int i = 0; i < meshSize.Length; i++) {
        double val = errNorms.dataGroups[1].Values[i];
        Console.WriteLine($"mesh{meshSize[i]}: {val}");
        NUnit.Framework.Assert.AreEqual(refl2Values[i], val, refl2Values_err[i], 
            $"l2-error for mesh{meshSize[i]} does not match reference value {refl2Values[i]}.");
    }

    Console.WriteLine($"EOC: {errNorms.Regression().ElementAt(1).Value}");
    NUnit.Framework.Assert.AreEqual(refEOC, errNorms.Regression().ElementAt(1).Value, refEOC_err, 
            $"Correspodning EOC does not match reference value {refEOC}.");
} 

## TABLE 2 - l2-error norms for the capillary wave studies

l2-errors on each grid ($\lambda / h = \{ 8, 16, 32\}$) and corresponding EOC for $La = 3$ 

In [108]:
var errNorms = GetErrorNormForConvergence_Plot2Ddata(evalData, evalSess, dataRef, "La3");

In [109]:
Checkl2errorNorms(errNorms, new double[] { 2.567e-2, 3.962e-3, 1.067e-3 }, new double[] { 0.005, 0.0004, 0.0002 }, 2.268, 0.2);

l2-errors on each grid ($\lambda / h = \{ 8, 16, 32\}$) and corresponding EOC for $La = 1.2 \cdot 10^2$ 

In [110]:
var errNorms = GetErrorNormForConvergence_Plot2Ddata(evalData, evalSess, dataRef, "La120");

In [111]:
Checkl2errorNorms(errNorms, new double[] { 2.954e-1, 1.051e-2, 3.487e-3 }, new double[] { 0.3, 0.005, 0.0004 }, 3.167, 1.2);

l2-errors on each grid ($\lambda / h = \{ 8, 16, 32\}$) and corresponding EOC for $La = 3\cdot 10^3$ 

In [112]:
var errNorms = GetErrorNormForConvergence_Plot2Ddata(evalData, evalSess, dataRef, "La3000");

In [113]:
Checkl2errorNorms(errNorms, new double[] { 8.314e-2, 2.700e-2, 1.021e-2 }, new double[] { 0.03, 0.004, 0.0003 }, 1.495, 0.2);

l2-errors on each grid ($\lambda / h = \{ 8, 16, 32\}$) and corresponding EOC for $La = 3 \cdot 10^5$ 

In [114]:
var errNorms = GetErrorNormForConvergence_Plot2Ddata(evalData, evalSess, dataRef, "La3e5");

In [116]:
Checkl2errorNorms(errNorms, new double[] { 1.713-1, 7.827e-2, 2.084e-2 }, new double[] { 0.6, 0.02, 0.0007 }, 1.501, 0.4);

l2-errors on each grid ($\lambda / h = \{ 8, 16, 32\}$) and corresponding EOC for $La = 3 \cdot 10^5$ with fixed dt

In [117]:
var errNorms = GetErrorNormForConvergence_Plot2Ddata(evalData, evalSess, dataRef, "La3e5_dtfix");

In [119]:
Checkl2errorNorms(errNorms, new double[] { 1.740-1, 4.044e-2, 1.288e-2 }, new double[] { 0.6, 0.02, 0.0005 }, 1.856, 0.2);

l2-errors on each grid ($\lambda / h = \{ 8, 16, 32\}$) and corresponding EOC for $La = 3 \cdot 10^5$ with fixed dt and $k=3$

In [120]:
var errNorms = GetErrorNormForConvergence_Plot2Ddata(evalData, evalSess, dataRef, "La3e5_dtfix", 3);

In [121]:
Checkl2errorNorms(errNorms, new double[] { 5.068e-2, 8.168e-3, 7.789e-3 }, new double[] { 0.005, 0.0009, 0.0008 }, 1.336, 0.2);